# Merge biological overlap + ATC + Morgan similarity into one drug-pair feature table

Loads the three independently-computed similarity sources (biological overlap, ATC, Morgan
structural) and merges them on `(drug1_id, drug2_id)` -- not by row position, since each was
computed in a separate notebook and may not share row order -- then validates the merge.

In [1]:
import pandas as pd
import numpy as np
import random
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
import matplotlib.pyplot as plt
import os


In [2]:
adverse_bio_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h1_biological_overlap\adverse_biological_overlap_extended.parquet")
non_interacting_bio_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h1_biological_overlap\non_interacting_biological_overlap_extended.parquet")

adverse_atc_sim_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h2_pharmacological_similarity\adverse_atc_sim.parquet")
non_interacting_atc_sim_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h2_pharmacological_similarity\non_interacting_atc_sim.parquet")

adverse_morgan_sim_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h3_structural_similarity\adverse_structural_sim.parquet")
non_interacting_morgan_sim_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\h3_structural_similarity\non_interacting_structural_sim.parquet")

for label, df in [("bio (adverse)", adverse_bio_df), ("bio (non_int)", non_interacting_bio_df),
                   ("atc (adverse)", adverse_atc_sim_df), ("atc (non_int)", non_interacting_atc_sim_df),
                   ("morgan (adverse)", adverse_morgan_sim_df), ("morgan (non_int)", non_interacting_morgan_sim_df)]:
    n_drugs = len(set(df['drug1_id']) | set(df['drug2_id']))
    print(f"{label:<20}: {len(df):>8,} rows | {n_drugs:>6,} unique drugs")


bio (adverse)       :  478,324 rows |  1,900 unique drugs
bio (non_int)       :  162,893 rows |  1,902 unique drugs
atc (adverse)       :  478,324 rows |  1,900 unique drugs
atc (non_int)       :  162,893 rows |  1,902 unique drugs
morgan (adverse)    :  478,324 rows |  1,900 unique drugs
morgan (non_int)    :  162,893 rows |  1,902 unique drugs


In [3]:
def merge_on_drug_pair(base_df, other_df, other_cols, label):
    """
    Merge `other_df[other_cols]` onto `base_df` keyed by (drug1_id, drug2_id), safely handling
    duplicate sampled pairs via an `_occurrence` rank within each (drug1_id, drug2_id) group so
    duplicates pair up 1st-with-1st, 2nd-with-2nd, etc. instead of a cartesian blow-up.
    """
    base = base_df.reset_index(drop=True).copy()
    other = other_df.reset_index(drop=True).copy()

    base['_occurrence'] = base.groupby(['drug1_id', 'drug2_id']).cumcount()
    other['_occurrence'] = other.groupby(['drug1_id', 'drug2_id']).cumcount()

    merged = base.merge(
        other[['drug1_id', 'drug2_id', '_occurrence'] + other_cols],
        on=['drug1_id', 'drug2_id', '_occurrence'],
        how='left',
        validate='one_to_one',
    )

    unmatched = merged[other_cols[0]].isna().sum()
    if unmatched:
        raise ValueError(f"{label}: {unmatched:,} / {len(merged):,} rows had no matching (drug1_id, drug2_id) in {label}")
    if len(merged) != len(base):
        raise ValueError(f"{label}: row count changed during merge ({len(base):,} -> {len(merged):,})")

    return merged.drop(columns=['_occurrence'])


def build_unified_pair_df(bio_df, atc_df, morgan_df):
    unified = merge_on_drug_pair(bio_df, atc_df, ['atc_similarity_score'], 'atc')
    unified = merge_on_drug_pair(unified, morgan_df, ['similarity_score'], 'morgan')
    unified = unified.rename(columns={'atc_similarity_score': 'atc_sim', 'similarity_score': 'structural_sim'})
    return unified


unified_adverse_df = build_unified_pair_df(adverse_bio_df, adverse_atc_sim_df, adverse_morgan_sim_df)
unified_non_interacting_df = build_unified_pair_df(non_interacting_bio_df, non_interacting_atc_sim_df, non_interacting_morgan_sim_df)

print(f"unified_adverse_df: {unified_adverse_df.shape}")
print(f"unified_non_interacting_df: {unified_non_interacting_df.shape}")


unified_adverse_df: (478324, 31)
unified_non_interacting_df: (162893, 31)


In [4]:
# Independent re-check: for a random sample of pairs, look up atc_sim/structural_sim directly by
# (drug1_id, drug2_id) in the ORIGINAL source frames and confirm they match what landed in the
# unified frame -- this catches a correct-looking merge that still silently mismatched rows.
def spot_check(unified_df, atc_df, morgan_df, label, n=20):
    atc_lookup = atc_df.groupby(['drug1_id', 'drug2_id'])['atc_similarity_score'].apply(list).to_dict()
    morgan_lookup = morgan_df.groupby(['drug1_id', 'drug2_id'])['similarity_score'].apply(list).to_dict()

    sample_idx = random.sample(range(len(unified_df)), n)
    bad = 0
    for i in sample_idx:
        row = unified_df.iloc[i]
        key = (row['drug1_id'], row['drug2_id'])
        if row['atc_sim'] not in atc_lookup.get(key, []) or row['structural_sim'] not in morgan_lookup.get(key, []):
            bad += 1
    print(f"{label}: {n - bad}/{n} sampled rows verified correct against source frames")
    assert bad == 0, f"{label}: {bad} sampled rows did NOT match their source (drug1_id, drug2_id) values"

spot_check(unified_adverse_df, adverse_atc_sim_df, adverse_morgan_sim_df, 'adverse')
spot_check(unified_non_interacting_df, non_interacting_atc_sim_df, non_interacting_morgan_sim_df, 'non_interacting')

print("\nNaNs introduced by merge:")
print(unified_adverse_df[['atc_sim', 'structural_sim']].isna().sum())
print(unified_non_interacting_df[['atc_sim', 'structural_sim']].isna().sum())


adverse: 20/20 sampled rows verified correct against source frames
non_interacting: 20/20 sampled rows verified correct against source frames

NaNs introduced by merge:
atc_sim           0
structural_sim    0
dtype: int64
atc_sim           0
structural_sim    0
dtype: int64


In [5]:
unified_adverse_df.to_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\unified_adverse_df.parquet")
unified_non_interacting_df.to_parquet(r"C:\Users\ashto\ddi-prediction\notebooks\unified_non_interacting_df.parquet")
print("Saved unified drug-pair feature tables.")


Saved unified drug-pair feature tables.
